In [ ]:
import sys
import torch
import random
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Add project root to path for modular imports
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Required library:
# !pip install tf-keras

# 2. Import custom modules to handle the data fetching and processing
from src.data_loader import DataOrchestrator
from src.data_processor import FeatureProcessor

# 3. Enable autoreload to catch changes in .py files instantly
%load_ext autoreload
%autoreload 2

print(" Environment Ready.")

In [ ]:
# --- 1. Stocks Selection ---

# You can change this list to any N stocks you want to experiment with
STOCKS_LIST = [
    "AAPL", "ABBV", "ADBE", "AMD", "AMT", "AMZN", "AVGO", "BAC", "BHP", "BLK",
    "BP", "COP", "COST", "CVX", "DLR", "EQIX", "FCX", "GOOGL", "GS", "INTC",
    "JNJ", "JPM", "LIN", "LLY", "META", "MRK", "MS", "NEM", "NFLX", "NVDA",
    "O", "PEP", "PFE", "PLD", "QCOM", "RIO", "SCCO", "SCHW", "SHW", "SLB",
    "SPG", "TMO", "TSLA", "TTE", "UNH", "WELL", "WFC", "XOM"
]

# --- 2. Hyperparameters ---

CONFIG = {
    'M': 55,                # Lookback window
    'T': 5,                 # Prediction horizon
    'ATR_MULTIPLIER': 1.5,  # For labeling threshold
    'NEWS_THRESHOLD': 10,   # How many top stocks to include in the fused dataset
}

# --- 3. API Keys & Ranges ---
API_KEYS = {
    'tiingo': 'YOUR_API',
    'alpha_vantage': 'YOUR_API',
    'eodhd': 'YOUR_API',
}

DATES = {
    'start': "2019-01-01", # Buffer for technical indicators
    'end': "2025-12-31"
}

print(f"Configured for {len(STOCKS_LIST)} stocks with M={CONFIG['M']}, T={CONFIG['T']}.")

Step 1: Sync Raw Data - Skip this cell if you already have the raw data

In [ ]:
print("Step 1: Syncing Raw Data...")
loader = DataOrchestrator(API_KEYS, DATES)
loader.sync_all(STOCKS_LIST)

Step 2: Process Datasets

In [ ]:
# --- Technical datasets ---

print("\n Step 2: Processing Datasets...")
generator = FeatureProcessor()

# 1. Generate the Technical + ER
generator.generate_tech_er_dataset(
    stocks=STOCKS_LIST,
    dates={'start': DATES['start'], 'end': DATES['end']},
    M=CONFIG['M'],
    T=CONFIG['T'],
    multiplier=CONFIG['ATR_MULTIPLIER']
)

In [ ]:
# --- Fused datasets, including news - skip if you don't have news data

# 2. Generate the Technical + ER + News Dataset
generator.generate_tech_er_news_dataset(
    stocks=STOCKS_LIST,
    dates={'start': DATES['start'], 'end': DATES['end']},
    M=CONFIG['M'],
    T=CONFIG['T'],
    multiplier=CONFIG['ATR_MULTIPLIER'],
    news_threshold=CONFIG['NEWS_THRESHOLD']
)